## Teoría base (cómo leer los bloques de código)

Aquí el modelo corre en local (Ollama) y responde token a token.

En cada bloque importante:
1. define prompt,
2. ejecuta inferencia,
3. evalúa formato/calidad de respuesta.

Pregunta guía: ¿el prompt controla realmente la salida o hace falta más estructura?

## Guía de lectura (teoría ↔ práctica)

Este cuaderno usa un LLM local vía Ollama para inferencia.

Idea central:
- el modelo define $P(y\mid x)$,
- el runtime local ejecuta generación token a token,
- la calidad depende de prompt + estrategia de decoding.

**Qué observar**: latencia, formato de respuesta y reproducibilidad con prompts similares.

# Ollama and Gemini API

## Ollama local

In [ ]:
import ollama

response = ollama.chat(
    model="hdnh2006/salamandra-2b-instruct:latest",
    messages=[
        {"role": "system", "content": "Eres un asistente maravilloso de programación."},
        {"role": "user", "content": "Escribe una función en Python para búsqueda binaria para un array ordenado."}
    ],
    options={
        "temperature": 0.3
    }
)

print(response["message"]["content"])

## Ollama cloud

In [ ]:
from ollama import Client

API_KEY = "<INSERT_YOUR_API_KEY_HERE>"

client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + API_KEY}
)

messages = [
  {"role": "system", "content": "Eres un asistente maravilloso de programación."},
  {"role": "user", "content": "Escribe una función en Python para búsqueda binaria para un array ordenado."},
]

response = client.chat('ministral-3:14b-cloud', messages=messages, options={"temperature": 0.3})
print(response['message']['content'])

## Gemini


In [ ]:
from google import genai
import os

os.environ["GEMINI_API_KEY"] = "<token>"
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Escribe una función en Python para búsqueda binaria para un array ordenado.",
    config={
        "system_instruction": "Eres un gran programador, experto en Python. Usar emojis en los comentarios para ser friendly.",
    }
)
print(response.text)

# OpenAI API

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="TOKEN")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain quicksort and give Python code."}
    ],
    temperature=0.2,
    max_tokens=2000
)

print(response.choices[0].message.content)

# Ejercicio

Haz un programa que reciba como entrada una petición de un usuario en lenguaje natural de generación de código en Python (ejemplo, "dame un programa que ordene un array") y devuelva un programa en Python que cumpla esa función. Para ello, haz lo siguiente:
1. Crea un prompt adecuado y mete en él la petición del usuario.
2. Llama a alguna API (Ollama, Gemini, etc.) para generar el código.
3. Extrae el código con regex.
4. Ejecuta el código generado. Si da errores, vuelve a llamar al LLM con el feedback. Para ejecutar código en Python, puedes usar la función `exec`.
```python
import traceback

codigo_generado = "1/0"
try:
    exec(codigo_generado)
except Exception as e:
    error_completo = traceback.format_exc()
    print(error_completo)
```
5. Repite el proceso hasta que el código se ejecute sin errores con un máximo de 5 iteraciones.

In [2]:
import re
import traceback
import textwrap
import ollama

MODEL_NAME = "qwen2.5:0.5b"
MAX_ITERS = 5
DEFAULT_REQUEST = "dame un programa que ordene un array"

try:
    user_request = input("Introduce tu petición de generación de código en Python: ").strip()
except EOFError:
    user_request = ""

if not user_request:
    user_request = DEFAULT_REQUEST

print(f"Petición usada: {user_request}")


def build_messages(request: str, previous_code: str | None = None, error_feedback: str | None = None):
    system_prompt = (
        "Eres un asistente experto en Python. "
        "Devuelve SIEMPRE código Python ejecutable dentro de un bloque ```python ... ```. "
        "No añadas explicación fuera del bloque de código."
    )

    if previous_code is None:
        user_prompt = (
            f"Genera un programa Python para esta petición: {request}.\n"
            "El programa debe poder ejecutarse directamente."
        )
    else:
        user_prompt = (
            f"La petición original era: {request}.\n"
            "Tu código anterior falló al ejecutarse.\n"
            "Código anterior:\n"
            f"```python\n{previous_code}\n```\n\n"
            "Error completo:\n"
            f"```\n{error_feedback}\n```\n\n"
            "Devuelve una versión corregida, completa y ejecutable."
        )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def extract_python_code(llm_text: str) -> str:
    pattern = r"```python\s*(.*?)```"
    match = re.search(pattern, llm_text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()

    fallback = llm_text.strip()
    if fallback.startswith("```") and fallback.endswith("```"):
        fallback = fallback.strip("`").strip()
    return fallback


def execute_generated_code(code: str):
    namespace = {}
    try:
        exec(code, namespace, namespace)
        return True, None
    except Exception:
        return False, traceback.format_exc()


last_code = None
last_error = None

for i in range(1, MAX_ITERS + 1):
    print(f"\n--- Iteración {i}/{MAX_ITERS} ---")

    response = ollama.chat(
        model=MODEL_NAME,
        messages=build_messages(user_request, previous_code=last_code, error_feedback=last_error),
        options={"temperature": 0.2},
    )

    llm_text = response["message"]["content"]
    code = extract_python_code(llm_text)
    last_code = code

    print("Código generado:\n")
    print(textwrap.indent(code, prefix="    "))

    ok, error = execute_generated_code(code)
    if ok:
        print("\n✅ Código ejecutado sin errores.")
        break

    last_error = error
    print("\n❌ Error al ejecutar. Se reintenta con feedback:\n")
    print(last_error)
else:
    print("\n⚠️ Se alcanzó el máximo de iteraciones sin ejecución correcta.")

Petición usada: Dame un programa python con un for que ordene 15 arrays y esas arrays las ordene segun el mayor numero de esas arrays

--- Iteración 1/5 ---
Código generado:

    import random

    # Definir los arrays con números aleatorios entre 1 y 50
    arrays = [
        [random.randint(1, 50), random.randint(1, 50)],
        [random.randint(1, 50), random.randint(1, 50)],
        [random.randint(1, 50), random.randint(1, 50)],
        [random.randint(1, 50), random.randint(1, 50)],
        [random.randint(1, 50), random.randint(1, 50)]
    ]

    # Ordenar los arrays
    arrays.sort(key=lambda x: max(x))

    # Mostrar los arrays ordenados
    for array in arrays:
        print(array)
[25, 7]
[33, 6]
[35, 32]
[2, 40]
[29, 46]

✅ Código ejecutado sin errores.
